In [1]:
import os

data_dir = "../data/raw"
documents = {}

for filename in os.listdir(data_dir):
    if filename.endswith(".md"):
        with open(os.path.join(data_dir,filename),"r", encoding="utf-8") as f:
            documents[filename] = f.read()

print(f"Loaded {len(documents)} files")
print(list(documents.keys()))
print(documents[list(documents.keys())[0]][:300])            

Loaded 3 files
['eviction-laws-and-procedures.md', 'rent-control-and-lease-agreements.md', 'tenant-rights-and-obligations.md']
# Karnataka Eviction Laws and Procedures

## Overview

Eviction of tenants in Karnataka is governed by the **Karnataka Rent Act, 1999**, the **Transfer of Property Act, 1882**, and applicable provisions of the **Code of Civil Procedure, 1908**. A landlord cannot evict a tenant without following due 


## Chunking


In [2]:
import re

def parse_markdown_structure(text, filename):
    lines = text.split('\n')
    chunks = []

    #track current heading at each level
    heading_stack = {}
    body = []
    in_code_block = False

    def flush():
        content = "\n".join(body).strip()
        if content:
            path = " > ".join(
                heading_stack[level] for level in sorted(heading_stack.keys())
            )
            chunks.append({
                "title": path if path else "Untitled",
                "text": content,
                "source": filename
            })
        body.clear()

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("```"):
            in_code_block = not in_code_block
            body.append(line)
            continue
        if in_code_block:
            body.append(line)
            continue

        heading_match = re.match(r'^(#{1,6})\s+(.*)', stripped)
        if heading_match:
            flush()
            level = len(heading_match.group(1))
            title = heading_match.group(2).strip()
            heading_stack[level] = title
            for deeper in list(heading_stack.keys()):
                if deeper > level:
                    del heading_stack[deeper]
        elif stripped == "---":
            continue
        else:
            body.append(line)
    flush()
    return chunks

In [3]:
all_chunks = []

for filename, text in documents.items():
    chunks = parse_markdown_structure(text, filename)
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
for c in all_chunks[:5]:
    print(f"[{c['title']}]\n{c['text'][:150]}...\n{'-'*40}")

Total chunks: 91
[Karnataka Eviction Laws and Procedures > Overview]
Eviction of tenants in Karnataka is governed by the **Karnataka Rent Act, 1999**, the **Transfer of Property Act, 1882**, and applicable provisions of...
----------------------------------------
[Karnataka Eviction Laws and Procedures > 1. Grounds for Eviction Under the Karnataka Rent Act, 1999 > Section 21 — Grounds on Which a Landlord May Seek Eviction]
A landlord may file an eviction petition before the **Rent Authority** (or Civil Court where the Rent Authority is not constituted) on the following g...
----------------------------------------
[Karnataka Eviction Laws and Procedures > 1. Grounds for Eviction Under the Karnataka Rent Act, 1999 > Section 21 — Grounds on Which a Landlord May Seek Eviction > (a) Non-Payment of Rent]
- The tenant has failed to pay rent for a period of **two months** after it became due.
- The landlord must issue a **written notice** demanding payme...
-----------------------------------

In [4]:
def enforce_size_bounds(text, max_chars=800):
    if len(text) <= max_chars:
        return [text]
    # split oversized section by paragraphs, regrouping into ~max_chars pieces
    paragraphs = text.split('\n\n')
    result, buf = [], ""
    for p in paragraphs:
        if len(buf) + len(p) < max_chars:
            buf += p + "\n\n"
        else:
            result.append(buf.strip())
            buf = p + "\n\n"
    if buf:
        result.append(buf.strip())
    return result

final_chunks = []
for c in all_chunks:
    pieces = enforce_size_bounds(c["text"])
    for i, piece in enumerate(pieces):
        final_chunks.append({
            "title": c["title"] + (f" (part {i+1})" if len(pieces) > 1 else ""),
            "text": piece,
            "source": c["source"]
        })

print(f"Final chunk count after size enforcement: {len(final_chunks)}")

Final chunk count after size enforcement: 94


## Embedding


In [5]:
pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
texts = [c["text"] for c in final_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True)

print(f"Embedded {len(embeddings)} chunks")
print(f"Vector shape: {embeddings[0].shape}")

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedded 94 chunks
Vector shape: (384,)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "How much security deposit can a landlord ask for?"
query_vec = embedder.encode([query])

sims = cosine_similarity(query_vec, embeddings)[0]
top_idx = np.argsort(sims)[::-1][:3]  # top 3 most similar chunks

for i in top_idx:
    print(f"Score: {sims[i]:.3f} | {final_chunks[i]['title']}")
    print(final_chunks[i]['text'][:150])
    print("-" * 40)

Score: 0.630 | Karnataka Tenant Rights and Landlord Obligations > 1. Security Deposits > Allowable Deductions
A landlord may deduct from the security deposit only for:

1. **Unpaid rent**: Any rent remaining unpaid at the time the tenant vacates.
2. **Unpaid u
----------------------------------------
Score: 0.600 | Karnataka Tenant Rights and Landlord Obligations > 1. Security Deposits > Maximum Deposit Amounts
Under the Karnataka Rent Act, 1999, the security deposit is regulated as follows:

- **Residential premises**: The security deposit shall not exceed *
----------------------------------------
Score: 0.586 | Karnataka Rent Control and Lease Agreement Regulations > 7. Rent Payment During Disputes > Depositing Rent in Court
If a dispute arises between the landlord and tenant (e.g., the landlord refuses to accept rent, or there is a disagreement about the rent amount):

- 
----------------------------------------


## Store every embedding in ChromaDB

In [9]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [10]:
import chromadb

client = chromadb.PersistentClient(path="../vectorstore")
collection = client.get_or_create_collection(name="rental_law_docs")

In [11]:
ids = [f"chunk_{i}" for i in range(len(final_chunks))]
metadatas = [
    {"title": c["title"], "source": c["source"]}
    for c in final_chunks
]
documents_text = [c["text"] for c in final_chunks]

collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),   # convert numpy array to plain lists
    documents=documents_text,
    metadatas=metadatas
)

print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 94 chunks in ChromaDB


In [12]:
def retrieve(query, n_results=3):
    query_vec = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vec,
        n_results=n_results
    )
    return results
#this is the vector search
results = retrieve("How much security deposit can a landlord ask for?")

for i in range(len(results['documents'][0])):
    print(f"[{results['metadatas'][0][i]['title']}]")
    print(results['documents'][0][i][:150])
    print(f"(source: {results['metadatas'][0][i]['source']})")
    print("-" * 40)

[Karnataka Tenant Rights and Landlord Obligations > 1. Security Deposits > Allowable Deductions]
A landlord may deduct from the security deposit only for:

1. **Unpaid rent**: Any rent remaining unpaid at the time the tenant vacates.
2. **Unpaid u
(source: tenant-rights-and-obligations.md)
----------------------------------------
[Karnataka Tenant Rights and Landlord Obligations > 1. Security Deposits > Maximum Deposit Amounts]
Under the Karnataka Rent Act, 1999, the security deposit is regulated as follows:

- **Residential premises**: The security deposit shall not exceed *
(source: tenant-rights-and-obligations.md)
----------------------------------------
[Karnataka Rent Control and Lease Agreement Regulations > 7. Rent Payment During Disputes > Depositing Rent in Court]
If a dispute arises between the landlord and tenant (e.g., the landlord refuses to accept rent, or there is a disagreement about the rent amount):

- 
(source: rent-control-and-lease-agreements.md)
-----------------

## Send the most similar chunks to LLM for a structured output


In [16]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client_llm = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [14]:
def build_prompt(question, retrieved_results):
    context_blocks = []
    for i in range(len(retrieved_results['documents'][0])):
        title = retrieved_results['metadatas'][0][i]['title']
        text = retrieved_results['documents'][0][i]
        context_blocks.append(f"[{title}]\n{text}")
    
    context = "\n\n---\n\n".join(context_blocks)
    
    prompt = f"""You are a legal assistant helping a tenant understand their rental agreement under Karnataka rental law.

Use ONLY the context below to answer the question. If the context doesn't contain enough information to answer confidently, say so clearly instead of guessing.

When relevant, mention which section of law the answer comes from (use the bracketed titles as your reference).

Context:
{context}

Question: {question}

Answer:"""
    return prompt

In [20]:
def ask_rag(question, n_results=3):
    retrieved = retrieve(question, n_results=n_results)
    prompt = build_prompt(question, retrieved)
    
    response = client_llm.chat.completions.create(
        model="openai/gpt-oss-120b",  # currently available on standard Groq accounts
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response.choices[0].message.content
    return answer, retrieved

answer, retrieved = ask_rag("can i party with my friends till late night")
print(answer)

The rental‑agreement excerpts you provided do not contain a specific clause that addresses whether a tenant may hold late‑night parties for friends.  

What the context does say is:

* Under **“Peaceful and undisturbed enjoyment”** (Karnataka law and the Indian Easements Act, 1882) the tenant is entitled to use the premises without unreasonable interference.  
* The landlord **may not enter the premises at unreasonable hours without the tenant’s permission** and **repeated uninvited entries** can be treated as harassment (see the note on Section 441 of the Bharatiya Nyaya Sanhita, 2023).  

While these provisions protect the tenant’s right to enjoy the premises, they also imply that the tenant’s use must not create a disturbance that could be deemed unreasonable or harassing to the landlord or neighbours. In practice, most lease agreements include “quiet‑hours” or “no‑disturbance” clauses that limit noisy activities (such as late‑night parties) after a certain hour, but such a clause i